Clean items in weapons dataset

In [16]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("/workspaces/amc-research-sprint-lh_gl/data/amcdata_weapons_facilities_V2.csv", encoding = 'latin-1')

# clean item's names
df['item'] = df['item'].str.strip().str.lower()
df = df[df['item'].notna() & (df['item'] != '')]
df['item'] = df['item'].str.replace(',', " ")
df['item'] = df['item'].str.replace(r's$', '', regex=True)
df = df[df['summary_category'] != 1]

Standardizing empty values to NaN

In [17]:
df.replace([99, '99', 'N/A', '', ' '], np.nan, inplace=True)

,agreement_id,Item_letter,Item_number,item_type,item,facility,weapon_item_definition,c_weapon_item_definition,subcategory,subcategory_main,...,Unnamed: 492,Unnamed: 493,Unnamed: 494,Unnamed: 495,Unnamed: 496,Unnamed: 497,Unnamed: 498,Unnamed: 499,Unnamed: 500,Unnamed: 501
0,40,A,1.0,1.0,weapons general,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,50,A,1.0,1.0,nuclear weapon,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,50,B,2.0,3.0,radioactive waste,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,50,C,3.0,1.0,weapons general,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,90,A,1.0,1.0,nuclear weapon,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429,344,A,1.0,0.0,battle tank,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
430,344,B,2.0,0.0,armoured combat vehicle,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
431,344,C,3.0,0.0,artillery,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
432,344,D,4.0,0.0,combat aircraft,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Dropping columns not relevant to our research purposes

In [18]:
colunas_alvo = [
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal',
    'obligations_timeframe_type'
]

outras_colunas = [
    'agreement_id',
    'item_type',
    'item',
    'weapon_item_definition',
    'c_weapon_item_definition',
    'timeframe_set_time',
    'timeframe_other',
    'c_obligations_timeframe',
    'timeframe_phases',
    'c_timeframe_phases',
    'c_phases',
    'eliminitation',             # note: typo in the dataset
    'conversion',
    'modernization',
    'facility_destruction'
]

all_columns_to_keep = outras_colunas + colunas_alvo

df = df[all_columns_to_keep]

Identifying all unique item names

In [19]:
item_lst = list(df['item'].unique())
len(item_lst)

218

Loading a dual-use label dataset provided by Claude based on the Wassenaar Agreement and merging it with our dataset

In [20]:
dual_use_mapping_df = pd.read_csv('weapon_dual_use_labels.csv', encoding= 'latin-1')

df = pd.merge(df, dual_use_mapping_df, left_on = 'item', right_on = 'weapon_category', how = 'inner')

Merge description of items (weapon_item_definition and c_weapon_item_definition) for duplicated items

In [21]:
def merge_unique_text(series):
    vals = series.replace('', pd.NA).dropna().unique()
    return ' | '.join(sorted(vals)) if len(vals) > 0 else ''

# Apply to both description columns
for col in ['weapon_item_definition', 'c_weapon_item_definition']:
    df[col] = df.groupby('item')[col].transform(merge_unique_text)

print(df.head(15))

    agreement_id  item_type                                  item  \
0             40        1.0                       weapons general   
1             50        1.0                        nuclear weapon   
2             50        3.0                     radioactive waste   
3             50        1.0                       weapons general   
4             90        1.0                        nuclear weapon   
5             90        3.0                      fissile material   
6             70        1.0                        nuclear weapon   
7             70        3.0                      nuclear material   
8          70001        1.0                        nuclear weapon   
9          70002        1.0                        nuclear weapon   
10            80        1.0       objects carrying nuclear weapon   
11            80        1.0           weapons of mass destruction   
12            80        1.0                       weapons general   
13           100        1.0  summa

Combine agreement_ids for duplicated items

In [22]:
df['agreement_id'] = df['agreement_id'].astype(str)
df['agreement_id'] = df.groupby('item')['agreement_id'].transform(
    lambda x: ';'.join(sorted(x.replace('', pd.NA).dropna().astype(str).unique()))
)

Drop duplicates

In [23]:
df_no_dup = df.drop_duplicates(subset=['item'], keep='first')

Generates a clean dataset without duplicates

In [24]:
# df_exploded.to_csv('duplicates_result.csv', index=False)
df_no_dup.to_csv('duplicates_result.csv', index=False)

Split agreement_ids to match with agreement_info

In [25]:
df = df.drop_duplicates(subset=['item'], keep='first')

df['agreement_id'] = df['agreement_id'].str.split(';')
df_exploded = df.explode('agreement_id')
df_exploded['agreement_id'] = df_exploded['agreement_id'].str.strip()

Pulled agreement_info dataset

In [26]:
df_agr = pd.read_csv("/workspaces/amc-research-sprint-lh_gl/data/amcdata_agreement_info_V2.csv", encoding='latin-1')
df_weapons = df_exploded 

Selected which columns from each dataset to keep

In [27]:
weapons_col = [
    'agreement_id',
    'item_type',
    'item',
    'weapon_item_definition',
    'c_weapon_item_definition',
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal',
    'obligations_timeframe_type',
    'timeframe_set_time',
    'timeframe_other',
    'c_obligations_timeframe',
    'timeframe_phases',
    'c_timeframe_phases',
    'c_phases',
    'weapon_category',
    'dual_use',
    'rationale'
]

df_weapons = df_weapons[weapons_col]

agr_col = [
    'agreement_id',
    'year',
    'title_full',
    'title_short',
    'description',
    'agreement_nature',
    'superseeded_agreement',
    'region',
    'region_specific',
    'c_region_specific',
    'laterality',
    'format',
    'agreement_date',
    'status',
    'adoption',
    'adoption_date',
    'signed_definitive_signature',
    'signatory_states',
    'entered_into_force',
    'entry_into_force_date',
    'agreement_participants_nr',
    'agreement_participants',
    'nr_states_parties_total',
    'agreement_subject_to_reservations',
    'Instrument_reservations_deposited',
    'reservation_states_signature',
    'amendment',
    'c_amendment',
    'amendment_decision',
    'exit',
    'exit_other',
    'exit_notification',
    'state_withdrawal_nr',
    'state_withdrawal'
]

df_agr = df_agr[agr_col]

Turn agreement_id to string

In [28]:
df_agr['agreement_id'] = df_agr['agreement_id'].astype(str)
df_weapons['agreement_id'] = df_weapons['agreement_id'].astype(str)

Merge cleand and agreement_info datasets

In [29]:
df = df_weapons.merge(df_agr, on='agreement_id', how='left')

Generate a new excel file with merged data

In [30]:
df.to_csv('mergedresults.csv', index=False)